In [ ]:
import os
import nibabel as nib
import numpy as np
import pandas as pd
from scipy.stats import linregress

input_folder = "/home/amenacer/Stage/base_de_donnees/rats/img-segmented/"
output_csv = "/home/amenacer/Stage/base_de_donnees/rats/descripteurs_segmentation.csv"

summary = []

for i in range(1, 30):
    fname = f"rat{i}.nii.gz"
    path = os.path.join(input_folder, fname)
    if not os.path.exists(path):
        print(f"Manquant : {path}")
        continue
    img = nib.load(path)
    data = img.get_fdata()  # shape: (128, 128, t)
    n_frames = data.shape[-1]

    aires = []
    for t in range(n_frames):
        mask = data[..., t]
        aire = np.sum(mask > 0)
        aires.append(aire)

    aires = np.array(aires)
    aire_min = np.min(aires)
    aire_max = np.max(aires)
    aire_mean = np.mean(aires)
    aire_std = np.std(aires)
    
    # Pente ascendante (linéaire, sur toute la courbe)
    slope_total, _, _, _, _ = linregress(np.arange(n_frames), aires)
    
    # Option : pente "montée" et "descente" (max local)
    idx_min = np.argmin(aires)
    idx_max = np.argmax(aires)
    if idx_min < idx_max:
        # Pente montante : de min à max
        pente_asc = (aire_max - aire_min) / (idx_max - idx_min)
        pente_desc = (aire_min - aire_max) / (n_frames - idx_max + idx_min)
    else:
        # Si max précède min (cycle inversé)
        pente_asc = (aire_min - aire_max) / (idx_min - idx_max)
        pente_desc = (aire_max - aire_min) / (n_frames - idx_min + idx_max)
    
    summary.append({
        "rat": i,
        "aire_min": aire_min,
        "aire_max": aire_max,
        "aire_moy": aire_mean,
        "aire_std": aire_std,
        "pente_totale": slope_total,
        "pente_ascendante": pente_asc,
        "pente_descendante": pente_desc
    })

# Sauvegarde CSV récapitulatif
df_summary = pd.DataFrame(summary)
df_summary.to_csv(output_csv, index=False)
print(f"Descripteurs sauvegardés dans : {output_csv}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

input_folder = "/home/amenacer/Stage/base_de_donnees/rats/img-segmented/"
plots_folder = "/home/amenacer/Stage/base_de_donnees/rats/plots/"
os.makedirs(plots_folder, exist_ok=True)

for i in range(1, 30):
    fname = f"rat{i}.nii.gz"
    path = os.path.join(input_folder, fname)
    try:
        import nibabel as nib
        img = nib.load(path)
        data = img.get_fdata()
        n_frames = data.shape[-1]
        aires = [np.sum(data[..., t] > 0) for t in range(n_frames)]
        plt.figure()
        plt.plot(range(n_frames), aires, marker='o')
        plt.title(f"Rat {i} – Aire vs Temps")
        plt.xlabel("Frame (temps)")
        plt.ylabel("Aire du masque")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(plots_folder, f"rat{i}_aire_vs_temps.png"))
        plt.close()
    except Exception as e:
        print(f"Rat {i} : {e}")

print(f"Courbes sauvegardées dans : {plots_folder}")


In [ ]:
import os
import nibabel as nib
import numpy as np
import pandas as pd

input_folder = "/home/amenacer/Stage/base_de_donnees/rats/img-segmented/"
output_csv = "/home/amenacer/Stage/base_de_donnees/rats/aires_par_frame.csv"

all_data = []

for i in range(1, 30):
    fname = f"rat{i}.nii.gz"
    path = os.path.join(input_folder, fname)
    if not os.path.exists(path):
        print(f"Manquant : {path}")
        continue
    img = nib.load(path)
    data = img.get_fdata()  # shape: (128, 128, t)
    n_frames = data.shape[-1]
    for t in range(n_frames):
        aire = np.sum(data[..., t] > 0)
        all_data.append({
            "rat": i,
            "frame": t,
            "aire": aire
        })

df = pd.DataFrame(all_data)
df.to_csv(output_csv, index=False)
print(f"CSV global sauvegardé sous : {output_csv}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

csv_path = "/home/amenacer/Stage/base_de_donnees/rats/aires_par_frame.csv"
df = pd.read_csv(csv_path)

# 1️⃣ Statistiques descriptives globales et par rat
stats_global = df['aire'].describe()
print("Statistiques globales sur toutes les aires :")
print(stats_global)

stats_par_rat = df.groupby('rat')['aire'].describe()
print("\nStatistiques descriptives par rat :")
print(stats_par_rat)

# 2️⃣ Histogramme des aires global
plt.figure(figsize=(8,5))
plt.hist(df['aire'], bins=30, color='steelblue', edgecolor='black')
plt.title("Distribution globale des aires (toutes frames, tous rats)")
plt.xlabel("Aire")
plt.ylabel("Nombre d'observations")
plt.tight_layout()
plt.savefig("/home/amenacer/Stage/base_de_donnees/rats/analyse/histogramme_aires_global.png")
plt.close()

# 3️⃣ Boxplot par rat (variation inter-individuelle)
plt.figure(figsize=(12,6))
df.boxplot(column='aire', by='rat')
plt.title("Distribution des aires par rat")
plt.suptitle("")
plt.xlabel("Rat")
plt.ylabel("Aire")
plt.tight_layout()
plt.savefig("/home/amenacer/Stage/base_de_donnees/rats/analyse/boxplot_aires_par_rat.png")
plt.close()

# 4️⃣ Courbes Aire vs Temps pour chaque rat sur le même graphique
plt.figure(figsize=(14,7))
for rat in df['rat'].unique():
    sub = df[df['rat'] == rat]
    plt.plot(sub['frame'], sub['aire'], label=f"Rat {rat}", alpha=0.7)
plt.title("Aire vs temps (tous les rats)")
plt.xlabel("Frame (temps)")
plt.ylabel("Aire")
plt.legend(loc="upper right", ncol=2, fontsize=7)
plt.tight_layout()
plt.savefig("/home/amenacer/Stage/base_de_donnees/rats/analyse/aires_vs_temps_multi.png")
plt.close()

# 5️⃣ Pente linéaire (tendance) pour chaque rat
from scipy.stats import linregress
pentes = []
for rat in df['rat'].unique():
    sub = df[df['rat'] == rat]
    slope, intercept, r_value, p_value, std_err = linregress(sub['frame'], sub['aire'])
    pentes.append({'rat': rat, 'pente': slope, 'r2': r_value**2})
pentes_df = pd.DataFrame(pentes)
print("\nPente linéaire Aire vs Temps par rat :")
print(pentes_df)
pentes_df.to_csv("/home/amenacer/Stage/base_de_donnees/rats/analyse/pentes_par_rat.csv", index=False)

# 6️⃣ Statistiques globales exportées
stats_par_rat.to_csv("/home/amenacer/Stage/base_de_donnees/rats/analyse/stats_par_rat.csv")

print("\nAnalyse et graphiques sauvegardés dans /home/amenacer/rat/analyse/")
